In [ ]:
from pathlib import Path
import importlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
SOLVER_ROOT = WORK_ROOT / 'hyper_arc_solver'
SOLVER_ROOT.mkdir(parents=True, exist_ok=True)

print('=== 1. PRE-FLIGHT INPUT INVENTORY ===', flush=True)
input_files = sorted(path for path in INPUT_ROOT.rglob('*') if path.is_file())
for path in input_files[:200]:
    print(f' - {path}', flush=True)
if len(input_files) > 200:
    print(f' ... {len(input_files) - 200} additional files', flush=True)

print('=== 2. PREPARING SOLVER SOURCE ===', flush=True)
archives = sorted(INPUT_ROOT.rglob('solver_code.zip'))
if archives:
    print(f'Extracting {archives[0]}', flush=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(SOLVER_ROOT)
else:
    candidates = sorted(
        path for path in INPUT_ROOT.rglob('main.py')
        if (path.parent / 'hyper_arc').is_dir()
    )
    if not candidates:
        raise RuntimeError('Solver source missing. Attach a dataset containing solver_code.zip or main.py plus hyper_arc/.')
    source_root = candidates[0].parent
    shutil.copy2(source_root / 'main.py', SOLVER_ROOT / 'main.py')
    shutil.copytree(source_root / 'hyper_arc', SOLVER_ROOT / 'hyper_arc', dirs_exist_ok=True)

print('=== 3. VERIFYING PORTABLE DEPENDENCIES ===', flush=True)
import torch
try:
    importlib.import_module('geoopt')
except ImportError:
    wheels = sorted(INPUT_ROOT.rglob('geoopt-*-py3-none-any.whl'))
    if not wheels:
        raise RuntimeError('geoopt is unavailable. Attach its platform-independent py3-none-any wheel.')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', '--no-deps', str(wheels[0])],
        check=True,
    )

print('=== 4. SELECTING COMPETITION DATA ===', flush=True)
challenge_candidates = sorted(INPUT_ROOT.rglob('arc-agi_test_challenges.json'))
if not challenge_candidates:
    raise RuntimeError('ARC test challenges are missing. Attach the ARC Prize competition data.')
def challenge_rank(path):
    value = str(path).lower()
    if 'arc-prize-2026-arc-agi-2' in value:
        priority = 0
    elif 'arc-prize-2025' in value:
        priority = 1
    else:
        priority = 2
    return priority, len(path.parts), value
challenge_path = sorted(challenge_candidates, key=challenge_rank)[0]
print(f'Using {challenge_path}', flush=True)

print('=== 5. RUNNING CHECKPOINTED SOLVER ===', flush=True)
output_path = WORK_ROOT / 'submission.json'
checkpoint_path = WORK_ROOT / 'checkpoint.pt'
seed_bank_path = SOLVER_ROOT / 'hyper_arc' / 'seed_bank.json'
command = [
    sys.executable, str(SOLVER_ROOT / 'main.py'),
    '--challenge-file', str(challenge_path),
    '--output', str(output_path),
    '--checkpoint', str(checkpoint_path),
    '--seed-bank', str(seed_bank_path),
]
subprocess.run(command, cwd=SOLVER_ROOT, check=True)

print('=== 6. POST-FLIGHT SUBMISSION AUDIT ===', flush=True)
sys.path.insert(0, str(SOLVER_ROOT))
from main import load_challenges, validate_submission
submission = json.loads(output_path.read_text(encoding='utf-8'))
challenges = load_challenges(challenge_path)
validate_submission(submission, challenges)
print(f'SUCCESS: validated {len(submission)} tasks at {output_path}', flush=True)
